## Rio Olympics 2016 interactive medal comparison

In [1]:
import pandas as pd
from bokeh.io import output_notebook
output_notebook()
from ipywidgets import interact, IntSlider
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.transform import factor_cmap
from bokeh.palettes import Turbo256
athletes=pd.read_csv('../data/olympia2016_athletes.csv')
athletes.head()

Loading BokehJS ...

,id,name,nationality,sex,dob,height,weight,sport,gold,silver,bronze
0,736041664,A Jesus Garcia,ESP,male,10/17/69,1.72,64.0,athletics,0,0,0
1,532037425,A Lam Shin,KOR,female,9/23/86,1.68,56.0,fencing,0,0,0
2,435962603,Aaron Brown,CAN,male,5/27/92,1.98,79.0,athletics,0,0,1
3,521041435,Aaron Cook,MDA,male,1/2/91,1.83,80.0,taekwondo,0,0,0
4,33922579,Aaron Gate,NZL,male,11/26/90,1.81,71.0,cycling,0,0,0


In [2]:
# Group athletes by country
countries = athletes.groupby(
    'nationality',
    as_index=False
).agg(
    athletes=('id', 'count'),
    gold=('gold', 'sum'),
    silver=('silver', 'sum'),
    bronze=('bronze', 'sum')
)

# Calculate the total number of medals
countries['medals'] = (
    countries['gold']
    + countries['silver']
    + countries['bronze']
)

# Get maximum values
max_medals = int(countries['medals'].max())
max_athletes = int(countries['athletes'].max())

countries.head()

,nationality,athletes,gold,silver,bronze,medals
0,AFG,3,0,0,0,0
1,ALB,6,0,0,0,0
2,ALG,68,0,2,0,2
3,AND,5,0,0,0,0
4,ANG,26,0,0,0,0


In [3]:
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.plotting import figure, show
from bokeh.io import push_notebook
from ipywidgets import interact, IntSlider

# Create the initial data source
source = ColumnDataSource(countries.copy())

# Tooltip information
tooltips = [
    ('Country', '@nationality'),
    ('Athletes', '@athletes'),
    ('Total Medals', '@medals'),
    ('Gold', '@gold'),
    ('Silver', '@silver'),
    ('Bronze', '@bronze')
]

# Create the plot only once
plot = figure(
    title='Rio Olympics 2016 - Medal comparison',
    x_axis_label='Number of Medals',
    y_axis_label='Num of Athletes',
    width=700,
    height=450,
    tools='pan,wheel_zoom,box_zoom,reset,save'
)

plot.add_tools(HoverTool(tooltips=tooltips))

country_names = countries['nationality'].astype(str).tolist()

plot.scatter(
    x='medals',
    y='athletes',
    source=source,
    size=20,
    fill_color=factor_cmap(
        'nationality',
        palette=Turbo256[:len(country_names)],
        factors=country_names
    ),
    line_color='white',
    line_width=0.5,
    fill_alpha=0.65,
    hover_fill_alpha=1.0
)

plot.x_range.start = 0
plot.y_range.start = 0

# Display the plot and retain its notebook handle
handle = show(plot, notebook_handle=True)

medal_slider = IntSlider(
    min=0,
    max=max_medals,
    value=max_medals,
    step=1,
    orientation='horizontal',
    description='Max medals',
    continuous_update=False
)

athlete_slider = IntSlider(
    min=1,
    max=max_athletes,
    value=max_athletes,
    step=1,
    orientation='vertical',
    description='Max athletes',
    continuous_update=False
)

@interact(
    max_medals=medal_slider,
    max_athletes=athlete_slider
)
def update_plot(max_medals, max_athletes):
    filtered = countries[
        (countries['medals'] <= max_medals)
        & (countries['athletes'] <= max_athletes)
    ]

    source.data = ColumnDataSource.from_df(
        filtered.reset_index(drop=True)
    )

    push_notebook(handle=handle)

interactive(children=(IntSlider(value=264, continuous_update=False, description='Max medals', max=264), IntSli…